Imports & Drive Mount

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
print("Available GPUs:", gpus)

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPU Ready")

from tensorflow.keras.applications import DenseNet201
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.optimizers import Adam

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    balanced_accuracy_score,
    precision_recall_curve,
    average_precision_score
)

from sklearn.utils.class_weight import compute_class_weight

Dataset Path & Params

In [ ]:
train_dir = r'E:\ML\BDWaste_split\train'
val_dir   = r'E:\ML\BDWaste_split\validation'
test_dir  = r'E:\ML\BDWaste_split\test'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

Data Generator

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print("Class indices:", train_gen.class_indices)

Class Weights (from TRAIN split)

In [ ]:
classes = np.unique(train_gen.classes)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_gen.classes
)
class_weights = dict(zip(classes, weights))
print("Class weights:", class_weights)

Model: MobileNetV3-Large + Custom Head

In [ ]:
base_model = DenseNet201(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc')
    ]
)

model.summary()

Stage-1 Training (Head Only)

In [ ]:
checkpoint_stage1 = tf.keras.callbacks.ModelCheckpoint(
    'best_stage1.weights.h5',
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    save_weights_only=True
)

history_1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weights,
    callbacks=[checkpoint_stage1]
)

Stage-2 Fine-tuning (Safe)

In [ ]:
base_model.trainable = True

# Freeze BatchNorm layers for stability
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Unfreeze only the last N layers (you can tune N=20/30)
N = 20
for layer in base_model.layers[:-N]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc')
    ]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
]

history_2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks
)

Plot Training Curves (Loss, Accuracy, AUC)

In [ ]:
def plot_history(histories, metric, title):
    plt.figure(figsize=(7,5))
    for i, h in enumerate(histories, start=1):
        if metric in h.history and f"val_{metric}" in h.history:
            plt.plot(h.history[metric], label=f"Train ({i})")
            plt.plot(h.history[f"val_{metric}"], label=f"Val ({i})")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history([history_1, history_2], "loss", "Training vs Validation Loss")
plot_history([history_1, history_2], "accuracy", "Training vs Validation Accuracy")
plot_history([history_1, history_2], "auc", "Training vs Validation AUC")

Evaluation + Metrics

In [ ]:
y_true = test_gen.classes
y_prob = model.predict(test_gen).flatten()
y_pred = (y_prob > 0.5).astype(int)

loss, acc, auc_tf = model.evaluate(test_gen)

print("Test Accuracy:", acc)
print(f"\nOverall Accuracy: {acc:.4f}")
print(f"TensorFlow AUC:   {auc_tf:.4f}")

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=list(test_gen.class_indices.keys())
))

balanced_acc = balanced_accuracy_score(y_true, y_pred)
print(f"Balanced Accuracy: {balanced_acc:.4f}")

auc = roc_auc_score(y_true, y_prob)
print(f"AUC-ROC Score:     {auc:.4f}")

Confusion Matrix Plot (Heatmap)

In [ ]:
plt.figure(figsize=(5,4))
plt.imshow(cm, interpolation='nearest')
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(2)
class_names = list(val_gen.class_indices.keys())
plt.xticks(tick_marks, class_names, rotation=15)
plt.yticks(tick_marks, class_names)

thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()

ROC Curve Plot

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

Precision-Recall Curve

In [ ]:
prec, rec, _ = precision_recall_curve(y_true, y_prob)
ap = average_precision_score(y_true, y_prob)

plt.figure(figsize=(6,6))
plt.plot(rec, prec, label=f"AP = {ap:.2f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(True)
plt.show()